## 1. Importação de bibliotecas

In [1]:
#%pip install transformers accelerate kagglehub pandas numpy scikit-learn joblib -q
#%pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
#%pip install groq -q
import random
import kagglehub
import csv
import pandas as pd
import numpy as np
import torch
import os
from getpass import getpass
from groq import Groq
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib
import re

c:\Users\Usuario\Desktop\NOTEBOOK UPDATES\G9-BR-TEAM-18\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Verifica se a GPU está disponível
print("GPU disponível:", torch.cuda.is_available())

# 2. Mostra o nome da placa de vídeo (caso disponível)
if torch.cuda.is_available():
    print("Nome da GPU:", torch.cuda.get_device_name(0))
else:
    print("Usando CPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo sendo utilizado:", device)


GPU disponível: False
Usando CPU
Dispositivo sendo utilizado: cpu


## 2. Dados
Os dados foram gerados tentando manter uma proporção 33%/33%/33% com base no tipo de imóvel.

In [3]:
"""
Gerador de base de dados sintética para o problema de análise energética.
Gera apenas as colunas de entrada (features), com consumo_kwh correlacionado
de forma realista ao tipo de imóvel, quantidade de equipamentos e horas de
alto consumo, em vez de valores puramente aleatórios. Inclui também a
categoria de maior consumo, com distribuição ponderada conforme o tipo
de imóvel (comércios tendem a Refrigeração/Climatização).
"""

TIPOS_IMOVEL = ["Casa", "Apartamento", "Comercial"]
TARIFA_KWH = 0.75

CATEGORIAS_CONSUMO = ["Iluminação", "Refrigeração", "Climatização", "Eletrodomésticos", "Tecnologia", "Serviços", "Outros"]

# Comércios (Comercial/Industrial): Refrigeração e Climatização somam 70% de chance,
# os outros 5 dividem os 30% restantes (6% cada)
PESOS_COMERCIAL = [0.06, 0.35, 0.35, 0.06, 0.06, 0.06, 0.06]

# Residências (Casa/Apartamento): distribuição uniforme entre as 7 categorias
PESOS_RESIDENCIAL = [1 / 7] * 7


def sortear_categoria_consumo(tipo_imovel):
    if tipo_imovel in ("Comercial"):
        pesos = PESOS_COMERCIAL
    else:
        pesos = PESOS_RESIDENCIAL
    return random.choices(CATEGORIAS_CONSUMO, weights=pesos, k=1)[0]


def gerar_registro(id_cliente):
    tipo_imovel = random.choice(TIPOS_IMOVEL)
    quantidade_equipamentos = random.randint(1, 25)
    horas_alto_consumo = round(random.uniform(0, 12), 1)
    uso_horario_pico = random.random() < 0.5
    categoria_maior_consumo = sortear_categoria_consumo(tipo_imovel)

    # Consumo base varia conforme tipo de imóvel
    base_por_tipo = {
        "Casa": 250,
        "Apartamento": 150,
        "Comercial": 500,
    }[tipo_imovel]

    # Consumo cresce com equipamentos e horas de alto consumo, com ruído aleatório
    consumo_kwh = (
        base_por_tipo
        + quantidade_equipamentos * random.uniform(8, 15)
        + horas_alto_consumo * random.uniform(10, 20)
        + (50 if uso_horario_pico else 0)
        + random.gauss(0, 30)  # ruído
    )
    consumo_kwh = max(20, round(consumo_kwh, 1))

    return {
        "id_cliente": id_cliente,
        "consumo_kwh": consumo_kwh,
        "uso_horario_pico": uso_horario_pico,
        "quantidade_equipamentos": quantidade_equipamentos,
        "tipo_imovel": tipo_imovel,
        "horas_alto_consumo": horas_alto_consumo,
        "categoria_maior_consumo": categoria_maior_consumo,
    }


def gerar_base(n_registros=1000, caminho_saida="base_energetica.csv"):
    registros = [gerar_registro(id_cliente=i) for i in range(1, n_registros + 1)]

    with open(caminho_saida, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=registros[0].keys())
        writer.writeheader()
        writer.writerows(registros)

    print(f"Base gerada com {n_registros} registros em '{caminho_saida}'")
    return registros


if __name__ == "__main__":
    gerar_base(n_registros=1000, caminho_saida="base_energetica.csv")

Base gerada com 1000 registros em 'base_energetica.csv'


In [4]:
df = pd.read_csv('base_energetica.csv')

In [5]:
df["uso_horario_pico"] = df["uso_horario_pico"].astype(bool)
df.head()

,id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo,categoria_maior_consumo
0,1,322.9,True,6,Apartamento,2.4,Refrigeração
1,2,418.7,False,3,Casa,10.8,Tecnologia
2,3,594.8,True,10,Casa,10.9,Climatização
3,4,470.3,False,7,Casa,8.7,Iluminação
4,5,665.7,False,10,Comercial,5.2,Serviços


## 3. Exploração e limpeza dos dados

In [6]:
print("Dimensões:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)
print("\nValores nulos por coluna:")
print(df.isnull().sum())
print("\nRegistros duplicados:", df.duplicated().sum())

print("\nInformações adicionais:")
q33 = np.percentile(df['consumo_kwh'], 33)
q66 = np.percentile(df['consumo_kwh'], 66)
consumo_minimo = df['consumo_kwh'].min()
consumo_maximo = df['consumo_kwh'].max()
print('Consumo Minimo: ', consumo_minimo)
print('Consumo Maximo: ', consumo_maximo)
print('Percentil 33: ', q33)
print('Percentil 66: ', q66)

df.head().style.hide(axis="index")

Dimensões: (1000, 7)

Tipos de dados:
id_cliente                   int64
consumo_kwh                float64
uso_horario_pico              bool
quantidade_equipamentos      int64
tipo_imovel                    str
horas_alto_consumo         float64
categoria_maior_consumo        str
dtype: object

Valores nulos por coluna:
id_cliente                 0
consumo_kwh                0
uso_horario_pico           0
quantidade_equipamentos    0
tipo_imovel                0
horas_alto_consumo         0
categoria_maior_consumo    0
dtype: int64

Registros duplicados: 0

Informações adicionais:
Consumo Minimo:  145.9
Consumo Maximo:  1013.6
Percentil 33:  464.668
Percentil 66:  649.768


id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo,categoria_maior_consumo
1,322.900000,True,6,Apartamento,2.400000,Refrigeração
2,418.700000,False,3,Casa,10.800000,Tecnologia
3,594.800000,True,10,Casa,10.900000,Climatização
4,470.300000,False,7,Casa,8.700000,Iluminação
5,665.700000,False,10,Comercial,5.200000,Serviços


In [7]:
# Variáveis numéricas
print("Estatísticas descritivas")
display(df[["consumo_kwh", "quantidade_equipamentos", "horas_alto_consumo"]].describe())

# Variáveis categóricas/booleanas
print("\nDistribuição de tipo_imovel")
print(df["tipo_imovel"].value_counts())
print("\nDistribuição de uso_horario_pico")
print(df["uso_horario_pico"].value_counts(normalize=False))

Estatísticas descritivas


,consumo_kwh,quantidade_equipamentos,horas_alto_consumo
count,1000.000000,1000.00000,1000.000000
mean,565.366100,12.82400,5.955200
std,180.616235,7.15879,3.470065
min,145.900000,1.00000,0.000000
25%,422.050000,7.00000,3.000000
50%,546.150000,13.00000,6.050000
75%,709.650000,19.00000,8.825000
max,1013.600000,25.00000,12.000000



Distribuição de tipo_imovel
tipo_imovel
Comercial      342
Casa           337
Apartamento    321
Name: count, dtype: int64

Distribuição de uso_horario_pico
uso_horario_pico
True     500
False    500
Name: count, dtype: int64


## 4. Definição de categorias

In [8]:
"""
1. Rotula a base sintética criando um índice de ineficiência energética
   e cortando-o em quintis (Excelente / Bom / Mediano / Ruim / Crítico).
2. Treina e compara três classificadores: Regressão Logística, Random Forest
   e Árvore de Decisão.
3. Salva o melhor modelo (pipeline completo com pré-processamento).
"""

CAMINHO_ENTRADA = "base_energetica.csv"
CAMINHO_SAIDA = "base_energetica_rotulada.csv"

COLUNAS_NUMERICAS = ["consumo_kwh", "quantidade_equipamentos", "horas_alto_consumo"]
COLUNAS_CATEGORICAS = ["tipo_imovel"]
COLUNAS_BOOLEANAS = ["uso_horario_pico"]

CONSUMO_BASE_POR_TIPO = {
    "Casa": 250,
    "Apartamento": 150,
    "Comercial": 500,
}


def calcular_indice_ineficiencia(df):
    """Índice composto (0 a 1) combinando as variáveis de entrada disponíveis."""
    consumo_relativo = df["consumo_kwh"] / df["tipo_imovel"].map(CONSUMO_BASE_POR_TIPO)
    consumo_norm = (consumo_relativo / consumo_relativo.max()).clip(0, 1)
    equip_norm = (df["quantidade_equipamentos"] / df["quantidade_equipamentos"].max()).clip(0, 1)
    horas_norm = (df["horas_alto_consumo"] / df["horas_alto_consumo"].max()).clip(0, 1)
    pico_norm = df["uso_horario_pico"].astype(int)

    indice = (
        0.40 * consumo_norm
        + 0.25 * pico_norm
        + 0.20 * equip_norm
        + 0.15 * horas_norm
    )
    return indice


def rotular_por_quintis(indice):
    """Corta o índice em 5 faixas com quantidades iguais de cada classe."""
    return pd.qcut(
        indice,
        q=5,
        labels=["Excelente", "Bom", "Mediano", "Ruim", "Crítico"]
    )


def main():
    df = pd.read_csv(CAMINHO_ENTRADA)
    df["uso_horario_pico"] = df["uso_horario_pico"].astype(int)

    indice = calcular_indice_ineficiencia(df)
    df["categoria"] = rotular_por_quintis(indice)
    df.to_csv(CAMINHO_SAIDA, index=False)

    print("Distribuição das categorias:")
    print(df["categoria"].value_counts(), "\n")


if __name__ == "__main__":
    main()

Distribuição das categorias:
categoria
Excelente    200
Bom          200
Mediano      200
Ruim         200
Crítico      200
Name: count, dtype: int64 



In [9]:
df = pd.read_csv("base_energetica_rotulada.csv")
df.head()

,id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo,categoria_maior_consumo,categoria
0,1,322.9,1,6,Apartamento,2.4,Refrigeração,Mediano
1,2,418.7,0,3,Casa,10.8,Tecnologia,Excelente
2,3,594.8,1,10,Casa,10.9,Climatização,Crítico
3,4,470.3,0,7,Casa,8.7,Iluminação,Bom
4,5,665.7,0,10,Comercial,5.2,Serviços,Excelente


In [10]:
df['estimativa_financeira'] = df['consumo_kwh'] * TARIFA_KWH
df['consumo_kwh'] = df['consumo_kwh'].round(1)
df.head()

,id_cliente,consumo_kwh,uso_horario_pico,quantidade_equipamentos,tipo_imovel,horas_alto_consumo,categoria_maior_consumo,categoria,estimativa_financeira
0,1,322.9,1,6,Apartamento,2.4,Refrigeração,Mediano,242.175
1,2,418.7,0,3,Casa,10.8,Tecnologia,Excelente,314.025
2,3,594.8,1,10,Casa,10.9,Climatização,Crítico,446.100
3,4,470.3,0,7,Casa,8.7,Iluminação,Bom,352.725
4,5,665.7,0,10,Comercial,5.2,Serviços,Excelente,499.275


## 5. Treinamento do modelo de classificação

In [11]:
COLUNAS_NUMERICAS = ["consumo_kwh", "quantidade_equipamentos", "horas_alto_consumo"]
COLUNAS_CATEGORICAS = ["tipo_imovel"]  # adicione "categoria_maior_consumo" aqui, se for usar como feature
COLUNAS_BOOLEANAS = ["uso_horario_pico"]

pre_processador = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), COLUNAS_NUMERICAS + COLUNAS_BOOLEANAS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORICAS),
    ]
)

pipeline = Pipeline([
    ("pre", pre_processador),
    ("modelo", RandomForestClassifier(n_estimators=200, random_state=42)),
])

# Exclui id_cliente (sem valor preditivo) e estimativa_financeira (vazamento de dados,
# já que é apenas consumo_kwh * tarifa)
X = df[COLUNAS_NUMERICAS + COLUNAS_CATEGORICAS + COLUNAS_BOOLEANAS]
y = df["categoria"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline.fit(X_treino, y_treino)
y_pred = pipeline.predict(X_teste)

print(f"Acurácia: {accuracy_score(y_teste, y_pred):.3f}\n")
print(classification_report(y_teste, y_pred, zero_division=0))

joblib.dump(pipeline, "modelo_categorizacao.joblib")
print("\nModelo salvo em 'modelo_categorizacao.joblib'")

Acurácia: 0.925

              precision    recall  f1-score   support

         Bom       0.93      0.93      0.93        40
     Crítico       0.97      0.95      0.96        40
   Excelente       0.95      0.95      0.95        40
     Mediano       0.92      0.88      0.90        40
        Ruim       0.86      0.93      0.89        40

    accuracy                           0.93       200
   macro avg       0.93      0.93      0.93       200
weighted avg       0.93      0.93      0.93       200


Modelo salvo em 'modelo_categorizacao.joblib'


## 6. Importação do modelo para as recomendações

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
MODEL_NAME = "llama-3.3-70b-versatile"

if client:
    print("Cliente Groq configurado com o modelo:", MODEL_NAME)
else:
    print("Aviso: GROQ_API_KEY não definida — sem fallback Groq")

Cliente Groq configurado com o modelo: llama-3.3-70b-versatile
Modelo carregado: llama-3.3-70b-versatile


In [13]:
def gerar_recomendacoes(df: dict, categoria: str, max_new_tokens: int = 100) -> list[str]:
    """
    Gera 3 recomendações de eficiência energética com base nos dados de entrada
    e na categoria já calculada pelo classificador (Random Forest).
    """
    system_prompt = (
        "Você é um assistente especializado em eficiência energética residencial e comercial. "
        "Responda sempre em português do Brasil, de forma objetiva e sem rodeios."
    )

    regra_tom = (
        "Tom conforme a categoria: Excelente ou Bom = reforçar boas práticas já adotadas; "
        "Mediano = sugerir ajustes pontuais; Ruim ou Crítico = ser direto sobre a necessidade "
        "de mudança, com mais urgência em Crítico."
    )

    regra_categoria_consumo = (
        f"A categoria de maior consumo do imóvel é '{df['categoria_maior_consumo']}'. "
        f"Pelo menos uma das 3 recomendações deve ser voltada especificamente para reduzir "
        f"ou otimizar o consumo relacionado a essa categoria."
    )

    user_prompt = f"""Com base nos dados abaixo, gere exatamente 3 recomendações curtas, práticas
e realmente úteis para melhorar a eficiência energética do imóvel.

REGRAS OBRIGATÓRIAS:
- Envolva EXCLUSIVAMENTE: hábitos de uso de equipamentos elétricos, horários de consumo, ou
  manutenção/substituição de aparelhos elétricos. Nada de água, gás ou outros recursos.
- {regra_categoria_consumo}
- Baseie-se APENAS nos dados fornecidos. Não invente equipamentos ou hábitos não informados.
- Se o tipo de imóvel for "Apartamento", não sugira painéis solares ou soluções que dependam
  de telhado/espaço externo próprio.
- Cada recomendação deve abordar um aspecto diferente, sem repetir o mesmo tipo de dica.
- Não cite marcas, modelos ou preços. Não use termos técnicos sem explicação simples.
- Máximo 20 palavras por recomendação. Sem emojis, markdown ou numeração.
- {regra_tom}

Exemplo de estilo (não copie o conteúdo):
Reduzir o uso simultâneo de equipamentos durante o horário de pico.
Desligar aparelhos em modo stand-by quando não estiverem em uso.
Distribuir o uso de equipamentos de maior consumo ao longo do dia.

Dados do imóvel:
- Consumo mensal: {df['consumo_kwh']} kWh
- Uso em horário de pico: {"Sim" if df['uso_horario_pico'] else "Não"}
- Quantidade de equipamentos: {df['quantidade_equipamentos']}
- Tipo de imóvel: {df['tipo_imovel']}
- Horas de alto consumo por dia: {df['horas_alto_consumo']}
- Categoria de eficiência: {categoria}
- Categoria de maior consumo: {df['categoria_maior_consumo']}

Responda APENAS com as 3 recomendações, uma por linha, sem numeração,
sem introdução e sem comentários adicionais."""

    mensagens = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    resposta = client.chat.completions.create(
        model=MODEL_NAME,
        messages=mensagens,
        max_tokens=max_new_tokens,
        temperature=0,
    )

    texto = resposta.choices[0].message.content

    recomendacoes = [
        re.sub(r"^\s*[\d]+[\.\)]?\s*", "", linha).strip("-•* ").strip()
        for linha in texto.strip().split("\n")
        if linha.strip()
    ]
    return recomendacoes[:3]

In [14]:
modelo = joblib.load("modelo_categorizacao.joblib")

cliente_id = random.randint(1, len(df))
linha_cliente = df[df["id_cliente"] == cliente_id].iloc[0]
dados_cliente = linha_cliente[["consumo_kwh", "uso_horario_pico", "quantidade_equipamentos", "tipo_imovel", "horas_alto_consumo", "categoria_maior_consumo"]].to_dict()

# Agora a categoria vem do MODELO, não do índice de tercis
dados_cliente_df = pd.DataFrame([dados_cliente])
categoria = modelo.predict(dados_cliente_df)[0]
probabilidade = round(modelo.predict_proba(dados_cliente_df).max(), 2)

recomendacoes = gerar_recomendacoes(dados_cliente, categoria)

In [15]:
print(linha_cliente)
print(recomendacoes)
print(f"Categoria prevista: {categoria} (confiança: {probabilidade})")

id_cliente                          529
consumo_kwh                       562.6
uso_horario_pico                      1
quantidade_equipamentos              22
tipo_imovel                        Casa
horas_alto_consumo                  0.6
categoria_maior_consumo    Climatização
categoria                          Ruim
estimativa_financeira            421.95
Name: 528, dtype: object
['Desligar aparelhos de climatização quando não estiverem em uso.', 'Reduzir o uso simultâneo de equipamentos durante o horário de pico.', 'Substituir lâmpadas e aparelhos antigos por modelos mais eficientes.']
Categoria prevista: Ruim (confiança: 0.88)


### Exemplo saída comercial:
['1. Instale painéis solares fotovoltaicos.', '2. Faça um planejamento térmico da fachada do edifício.', '3. Optimize o gerenciamento da iluminação e das temperaturas interna.']

### Exemplo saída residencial (Casa)
['1. Reduza o número de equipamentos elétricos instalados.', '2. Implemente um plano de programação da luz para minimizar o uso na alta demanda.', '3. Opte por fontes alternativas de energia no fornecimento doméstico.']

### Exemplo de saída residencial (Apartamento)
id_cliente                         605
consumo_kwh                      289.1
uso_horario_pico                     1
**quantidade_equipamentos              1**
tipo_imovel                Apartamento
horas_alto_consumo                10.7
categoria                     Moderado
estimativa_financeira          216.825
Name: 604, dtype: object

['1. Desligue o equipamento principal em modo stand-by quando não estiver em uso.', '2. Distribua o uso do equipamento entre diferentes dias da semana para evitar o horário de pico.', '3. Faça uma análise detalhada do consumo diário para identificar áreas onde pode reduzir o uso excessivo.']


